In [1]:
import arcpy
import os
import re
import uuid
import pandas as pd

arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False

# -------------------------------------------------------------------
# PATHS
# -------------------------------------------------------------------
blocks_folder = r"C:\Users\CNEELY01\OneDrive - Environmental Protection Agency (EPA)\Documents\GitHub\Water-Infrastructure-and-Capacity-Assessment-Tool\R\Wastewater_Analysis\POTW_Blk_Crswlk\Temp_Outputs\blocks_2020_sf"
potw_points = r"C:\Users\CNEELY01\OneDrive - Environmental Protection Agency (EPA)\Documents\GitHub\Water-Infrastructure-and-Capacity-Assessment-Tool\R\Wastewater_Analysis\POTW_Blk_Crswlk\Temp_Outputs\Active_POTW_Coords.gdb\Active_POTW_Coords"
out_gdb = r"C:\Users\CNEELY01\OneDrive - Environmental Protection Agency (EPA)\Documents\GitHub\Water-Infrastructure-and-Capacity-Assessment-Tool\R\Wastewater_Analysis\POTW_Blk_Crswlk\Pro_Crsswlk_Files\POTW_Crosswalk.gdb"
temp_gdb = r"C:\Users\CNEELY01\OneDrive - Environmental Protection Agency (EPA)\Documents\GitHub\Water-Infrastructure-and-Capacity-Assessment-Tool\R\Wastewater_Analysis\POTW_Blk_Crswlk\Pro_Crsswlk_Files\POTW_Crosswalk_Temp.gdb"
export_folder = r"C:\Users\CNEELY01\OneDrive - Environmental Protection Agency (EPA)\Documents\GitHub\Water-Infrastructure-and-Capacity-Assessment-Tool\R\Wastewater_Analysis\POTW_Blk_Crswlk\Final_Crsswlk_Outputs"

os.makedirs(export_folder, exist_ok=True)

out_sr = arcpy.SpatialReference(5070)

# -------------------------------------------------------------------
# HELPERS
# -------------------------------------------------------------------
def safe_name(name, max_len=40):
    name = re.sub(r'[^A-Za-z0-9_]+', '_', name)
    return name[:max_len]

def unique_name(prefix):
    return f"{prefix}_{uuid.uuid4().hex[:8]}"

def export_table_to_csv(in_table, csv_path):
    fields = [f.name for f in arcpy.ListFields(in_table)
              if f.type not in ("Geometry", "Raster", "Blob")]
    rows = []
    with arcpy.da.SearchCursor(in_table, fields) as cursor:
        for row in cursor:
            rows.append([None if v is None else v for v in row])
    pd.DataFrame(rows, columns=fields).to_csv(csv_path, index=False)
    print(f"Exported CSV: {csv_path}")

def ensure_field(fc, field_name, field_type, field_length=None):
    existing = [f.name for f in arcpy.ListFields(fc)]
    if field_name not in existing:
        if field_type == "TEXT":
            arcpy.management.AddField(fc, field_name, field_type, field_length=field_length or 50)
        else:
            arcpy.management.AddField(fc, field_name, field_type)

def hard_fail_if_missing_npded_id(fc, context):
    missing = 0
    with arcpy.da.SearchCursor(fc, ["NPDES_ID"]) as cur:
        for (nid,) in cur:
            if nid in (None, ""):
                missing += 1
    if missing > 0:
        raise RuntimeError(f"{missing} rows missing NPDES_ID in {context}")

# -------------------------------------------------------------------
# CHECK / PREP
# -------------------------------------------------------------------
if not arcpy.Exists(temp_gdb):
    folder = os.path.dirname(temp_gdb)
    name = os.path.basename(temp_gdb)
    arcpy.management.CreateFileGDB(folder, name)

shp_files = [f for f in os.listdir(blocks_folder) if f.lower().endswith(".shp")]
if not shp_files:
    raise RuntimeError(f"No .shp files found in {blocks_folder}")

# -------------------------------------------------------------------
# PREP POTW POINTS
# -------------------------------------------------------------------
potw_lyr = f"potw_lyr_{uuid.uuid4().hex[:8]}"
arcpy.management.MakeFeatureLayer(potw_points, potw_lyr)

if "NPDES_ID" not in [f.name for f in arcpy.ListFields(potw_lyr)]:
    raise RuntimeError("POTW points do not contain NPDES_ID.")

potw_copy = arcpy.management.CopyFeatures(
    potw_lyr,
    os.path.join(temp_gdb, unique_name("POTW_Copy"))
)[0]

arcpy.management.DefineProjection(potw_copy, arcpy.SpatialReference(4326))

points_m = arcpy.management.Project(
    potw_copy,
    os.path.join(temp_gdb, unique_name("POTW_M")),
    out_sr
)[0]

# -------------------------------------------------------------------
# RUN ONE STATE
# -------------------------------------------------------------------
def process_state(shp):
    state_base = safe_name(os.path.splitext(shp)[0])
    src_shp = os.path.join(blocks_folder, shp)

    print(f"  Processing {shp}")

    # Import + project blocks
    imported_lyr = f"imp_lyr_{uuid.uuid4().hex[:8]}"
    arcpy.management.MakeFeatureLayer(src_shp, imported_lyr)

    imported_fc = arcpy.management.CopyFeatures(
        imported_lyr,
        os.path.join(temp_gdb, unique_name(f"imp_{state_base}"))
    )[0]

    # Make a fresh layer from imported FC before projecting (fix for read-only issue)
    imported_fc_lyr = f"impfc_lyr_{uuid.uuid4().hex[:8]}"
    arcpy.management.MakeFeatureLayer(imported_fc, imported_fc_lyr)

    proj_fc = arcpy.management.Project(
        imported_fc_lyr,
        os.path.join(temp_gdb, unique_name(f"proj_{state_base}")),
        out_sr
    )[0]

    blocks_fc = arcpy.management.CopyFeatures(
        proj_fc,
        os.path.join(temp_gdb, unique_name(f"blk_{state_base}"))
    )[0]

    ensure_field(blocks_fc, "block_area_m2", "DOUBLE")
    arcpy.management.CalculateGeometryAttributes(
        blocks_fc,
        [["block_area_m2", "AREA"]],
        area_unit="SQUARE_METERS"
    )

    # One 5-mile candidate block set per state
    state_env = arcpy.analysis.Buffer(
        blocks_fc,
        os.path.join(temp_gdb, unique_name(f"env_{state_base}")),
        "5 Miles",
        dissolve_option="ALL",
        method="PLANAR"
    )[0]

    potw_sel_lyr = f"potwsel_{uuid.uuid4().hex[:8]}"
    arcpy.management.MakeFeatureLayer(points_m, potw_sel_lyr)
    arcpy.management.SelectLayerByLocation(potw_sel_lyr, "INTERSECT", state_env)
    potw_sel_count = int(arcpy.management.GetCount(potw_sel_lyr)[0])

    if potw_sel_count == 0:
        print("    No POTW points near state envelope.")
        arcpy.management.Delete(potw_sel_lyr)
        return []

    potw_state = arcpy.management.CopyFeatures(
        potw_sel_lyr,
        os.path.join(temp_gdb, unique_name(f"potw_{state_base}"))
    )[0]
    arcpy.management.Delete(potw_sel_lyr)

    print(f"    POTW points selected for state: {potw_sel_count}")

    # 5-mi buffer and 5-mi candidate block set
    state_buf_5 = arcpy.analysis.Buffer(
        potw_state,
        os.path.join(temp_gdb, unique_name(f"buf_{state_base}_5mi")),
        "5 Miles",
        dissolve_option="NONE",
        method="PLANAR"
    )[0]

    arcpy.management.JoinField(state_buf_5, "ORIG_FID", potw_state, "OBJECTID", ["NPDES_ID"])

    cand_lyr_5 = f"cand5_{uuid.uuid4().hex[:8]}"
    arcpy.management.MakeFeatureLayer(blocks_fc, cand_lyr_5)
    arcpy.management.SelectLayerByLocation(cand_lyr_5, "INTERSECT", state_buf_5)
    cand5_count = int(arcpy.management.GetCount(cand_lyr_5)[0])

    print("    5-mi candidate blocks:", cand5_count)

    if cand5_count == 0:
        arcpy.management.Delete(cand_lyr_5)
        return []

    cand5_fc = arcpy.management.CopyFeatures(
        cand_lyr_5,
        os.path.join(temp_gdb, unique_name(f"cand5_{state_base}"))
    )[0]
    arcpy.management.Delete(cand_lyr_5)

    def run_buffer_on_candset(buffer_label, buffer_distance):
        print(f"    Running {buffer_label} from 5-mi candidate set...")

        state_buf = arcpy.analysis.Buffer(
            potw_state,
            os.path.join(temp_gdb, unique_name(f"buf_{state_base}_{buffer_label}")),
            buffer_distance,
            dissolve_option="NONE",
            method="PLANAR"
        )[0]

        arcpy.management.JoinField(state_buf, "ORIG_FID", potw_state, "OBJECTID", ["NPDES_ID"])

        cand_lyr = f"cand_{buffer_label}_{uuid.uuid4().hex[:8]}"
        arcpy.management.MakeFeatureLayer(cand5_fc, cand_lyr)
        arcpy.management.SelectLayerByLocation(cand_lyr, "INTERSECT", state_buf)
        cand_count = int(arcpy.management.GetCount(cand_lyr)[0])

        print(f"      {buffer_label} candidate blocks from 5-mi set:", cand_count)

        if cand_count == 0:
            arcpy.management.Delete(cand_lyr)
            return None

        cand_fc = arcpy.management.CopyFeatures(
            cand_lyr,
            os.path.join(temp_gdb, unique_name(f"cand_{state_base}_{buffer_label}"))
        )[0]
        arcpy.management.Delete(cand_lyr)

        ix_fc = arcpy.analysis.PairwiseIntersect(
            [cand_fc, state_buf],
            os.path.join(temp_gdb, unique_name(f"ix_{state_base}_{buffer_label}")),
            "ALL",
            None,
            "INPUT"
        )[0]

        ensure_field(ix_fc, "buffer_dist", "TEXT", 10)
        ensure_field(ix_fc, "overlap_m2", "DOUBLE")
        ensure_field(ix_fc, "decimal_overlap", "DOUBLE")
        ensure_field(ix_fc, "percent_overlap", "DOUBLE")

        arcpy.management.CalculateField(ix_fc, "buffer_dist", f"'{buffer_label}'", "PYTHON3")
        arcpy.management.CalculateGeometryAttributes(ix_fc, [["overlap_m2", "AREA"]], area_unit="SQUARE_METERS")

        with arcpy.da.UpdateCursor(ix_fc, ["overlap_m2", "block_area_m2", "decimal_overlap", "percent_overlap"]) as cur:
            for row in cur:
                ov = row[0] or 0
                ba = row[1]
                if ba and ba != 0:
                    row[2] = ov / ba
                    row[3] = (ov / ba) * 100
                else:
                    row[2] = None
                    row[3] = None
                cur.updateRow(row)

        hard_fail_if_missing_npded_id(ix_fc, f"{state_base} / {buffer_label}")

        csv_path = os.path.join(export_folder, f"{state_base}_{buffer_label}_from5miCand.csv")
        export_table_to_csv(ix_fc, csv_path)

        print(f"      {buffer_label} output rows:", int(arcpy.management.GetCount(ix_fc)[0]))
        return ix_fc

    out_tables = []
    out_5 = run_buffer_on_candset("5mi", "5 Miles")
    out_3 = run_buffer_on_candset("3mi", "3 Miles")
    out_1 = run_buffer_on_candset("1mi", "1 Miles")

    for t in [out_5, out_3, out_1]:
        if t:
            out_tables.append(t)

    return out_tables

# -------------------------------------------------------------------
# RESUME SUPPORT
# -------------------------------------------------------------------
resume_from = "KS"   # change to None to run from the beginning
if resume_from:
    start_index = next(i for i, shp in enumerate(shp_files) if shp.endswith(f"_{resume_from}.shp")) + 1
    shp_files_to_run = shp_files[start_index:]
    print(f"Resuming after {resume_from}; starting at index {start_index}")
else:
    shp_files_to_run = shp_files

# -------------------------------------------------------------------
# RUN ALL STATES / TERRITORIES
# -------------------------------------------------------------------
for shp in shp_files_to_run:
    process_state(shp)

print("\nDone.")
print("Outputs written to:", export_folder)

Resuming after KS; starting at index 19
  Processing blocks_2020_KY.shp
    POTW points selected for state: 436
    5-mi candidate blocks: 96638
    Running 5mi from 5-mi candidate set...
      5mi candidate blocks from 5-mi set: 96638
Exported CSV: C:\Users\CNEELY01\OneDrive - Environmental Protection Agency (EPA)\Documents\ArcGIS\Projects\Identifying Communities for TA_02\Exports\blocks_2020_KY_5mi_from5miCand.csv
      5mi output rows: 183776
    Running 3mi from 5-mi candidate set...
      3mi candidate blocks from 5-mi set: 64367
Exported CSV: C:\Users\CNEELY01\OneDrive - Environmental Protection Agency (EPA)\Documents\ArcGIS\Projects\Identifying Communities for TA_02\Exports\blocks_2020_KY_3mi_from5miCand.csv
      3mi output rows: 90981
    Running 1mi from 5-mi candidate set...
      1mi candidate blocks from 5-mi set: 19453
Exported CSV: C:\Users\CNEELY01\OneDrive - Environmental Protection Agency (EPA)\Documents\ArcGIS\Projects\Identifying Communities for TA_02\Exports\blocks